In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXX" # Add your namespace-prefix here

### Evalulate on a Single Video

We can identify the timestamps of when the VLM classifies something as having a broadcasting logo with the code below.

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
from pathlib import Path
import json

pop = Pop(components=[
   InferenceComponent(
       ability="eyepop.find-events.find-sponsor-logo:latest"
   )
])

in_logo_event = False
current_start_time = 0.0
last_seen_logo_end = 0.0
event_counter = 1

MAX_GAP_SECONDS = 0.5

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   sample_img_path = Path("/content/sample_video.mp4")
   job = endpoint.upload(sample_img_path)

   print("Processing video...\n")

   while result := job.predict():
        is_logo = False
        if "classes" in result and len(result["classes"]) > 0:
            if result["classes"][0].get("classLabel") == "logo":
                is_logo = True

        frame_sec = result.get("seconds", 0.0)
        frame_duration = result.get("duration", 0) / 1_000_000_000

        if is_logo:
            if not in_logo_event:
                in_logo_event = True
                current_start_time = frame_sec

            last_seen_logo_end = frame_sec + frame_duration

        else:
            if in_logo_event:
                time_since_last_logo = frame_sec - last_seen_logo_end

                if time_since_last_logo > MAX_GAP_SECONDS:
                    in_logo_event = False
                    total_duration = last_seen_logo_end - current_start_time

                    print(f"--- Logo Instance {event_counter} ---")
                    print(f"Start:    {current_start_time:.3f}s")
                    print(f"End:      {last_seen_logo_end:.3f}s")
                    print(f"Duration: {total_duration:.3f}s\n")

                    event_counter += 1

if in_logo_event:
    total_duration = last_seen_logo_end - current_start_time
    print(f"--- Logo Instance {event_counter} (Ended with video) ---")
    print(f"Start:    {current_start_time:.3f}s")
    print(f"End:      {last_seen_logo_end:.3f}s")
    print(f"Duration: {total_duration:.3f}s\n")

print("Done")